 # 데이터 전처리 및 파이프라인 구성
* 01 eda의 결과를 바탕으로 특성 선택 및 가공
* 전처리할 학습 세트는 01-eda에서 저장한 data/processed/01/train.csv 사용

0. 사용특성 선택
1. 숫자형 결측치 처리
2. 범주형 결측치 처리 및 인코딩
3. 특성 스케일링 및 분포 변환
4. 파이프라인 작성
5. 파이프라인 객체에 대해 학습셋으로 학습 후 테스트셋에 적용



In [63]:
import pandas as pd
train = pd.read_csv("../data/processed/01/train.csv").copy()

## 특성 선택
* Sex
* Pclass(숫자형처럼 생각하는게 더 나을듯 왜냐하면 1,2,3순서 자체에도 정보가 포함됨) -> Fare와 추이가 비슷하기 때문에 Pclass하나로 대체하는게 더 나을듯
* Sex + cat_age(숫자형) -> 그러나 나이대의 그룹을 나누는 기준은 다시 생각해봐야함 (깎아봐야앎)
* cat_parch(숫자형) -> 결측치 처리 -> 있냐 없냐 수준의 범주형으로 변경
* has_cabin(범주형) -> cabin에 대해 결측치 처리, 이진범주로 변경 해야함
* Title(범주형)


-> 근데? 일단은 베이스라인을 먼저 만들기 위해 연관이 약한 특성 제외 특성 가공을 빼고 일반 특성만 파이프라인에 집어넣어보자

## 숫자형 결측치 처리
-> SimpleImputer를 이용한 중간값 사용
* Parch, Age처리

In [64]:
from sklearn.impute import SimpleImputer

numbers = ["Parch", "Age", "Pclass"]
num_imputer = SimpleImputer(strategy="median")


## 숫자형 -> 범주형 변환

-> 일단 나중에 사용


In [65]:
train["cat_age"] = pd.cut(train["Age"], bins=[0, 15, 40, 100])

train["cat_parch"] = pd.cut(train["Parch"], bins=[0,1,10])
train["cat_parch"].value_counts()


cat_parch
(0, 1]     96
(1, 10]    77
Name: count, dtype: int64

## 범주형 결측치 처리

In [66]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categories = ["Sex"]
cat_imputer = SimpleImputer(strategy="most_frequent")
sex_encoder = OneHotEncoder()


### 파이프라인 생성

In [67]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline

sex_pipeline = make_pipeline(cat_imputer, sex_encoder)
num_pipeline = make_pipeline(num_imputer, StandardScaler())
pipeline = ColumnTransformer(
    [
        ("sex", sex_pipeline, ["Sex"]),
        ("num", num_imputer, numbers),
    ]
)

In [69]:
X = pipeline.fit_transform(train)
pd.DataFrame(X, columns=pipeline.get_feature_names_out())

,sex__Sex_female,sex__Sex_male,num__Parch,num__Age,num__Pclass
0,1.0,0.0,1.0,52.0,1.0
1,0.0,1.0,0.0,31.0,2.0
2,0.0,1.0,0.0,27.0,3.0
3,0.0,1.0,0.0,28.0,3.0
4,0.0,1.0,0.0,25.0,2.0
...,...,...,...,...,...
707,0.0,1.0,5.0,39.0,3.0
708,0.0,1.0,0.0,46.0,1.0
709,0.0,1.0,0.0,21.0,3.0
710,0.0,1.0,0.0,61.0,1.0
